# 电商用户流失预测与分群分析

**流程**：数据加载 → 划分与标准化 → 三模型对比（AUC）→ XGBoost 详解 → 特征重要性 → K-Means 分群 → 高风险清单输出

> 数据说明：模拟数据集（`generate_data.py` 生成 5000 条），分布假设贴近真实电商业务；
> 替换 `data/ecommerce_churn.csv` 为同结构真实数据即可复用全部建模代码。

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.cluster import KMeans

df = pd.read_csv("data/ecommerce_churn.csv")
df.head()

,user_id,recency,frequency,monetary,login_freq,browse_depth,addcart_rate,customer_service,promo_sensitive,churn
0,1,14.1,5,322.84,10,2,0.159,1,0.88,0
1,2,90.3,5,380.02,10,1,0.115,1,0.41,0
2,3,39.5,4,1034.33,14,35,0.145,1,0.14,0
3,4,27.4,9,410.95,12,4,0.364,0,0.03,0
4,5,5.1,2,461.36,15,35,0.048,1,0.69,0


## 1. 数据概览

流失率约 9%，类别不平衡——这就是后文用 **AUC 而非准确率** 评估的原因：全预测"不流失"准确率也有 91%，但毫无业务价值。

In [2]:
X = df.drop(columns=["user_id", "churn"])
y = df["churn"]

print(f"样本数 {len(df)}，流失率 {y.mean():.1%}")
df.describe().round(2)

样本数 5000，流失率 8.9%


,user_id,recency,frequency,monetary,login_freq,browse_depth,addcart_rate,customer_service,promo_sensitive,churn
count,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.0,5000.00,5000.00
mean,2500.50,29.69,5.02,507.36,10.01,25.01,0.25,1.0,0.50,0.09
std,1443.52,29.61,2.21,280.61,3.19,14.13,0.14,1.0,0.29,0.29
min,1.00,0.00,0.00,0.26,1.00,1.00,0.00,0.0,0.00,0.00
25%,1250.75,8.40,3.00,293.11,8.00,13.00,0.13,0.0,0.25,0.00
50%,2500.50,20.80,5.00,497.67,10.00,25.00,0.25,1.0,0.49,0.00
75%,3750.25,41.40,6.00,704.12,12.00,37.00,0.38,2.0,0.75,0.00
max,5000.00,245.20,15.00,1607.49,25.00,49.00,0.50,6.0,1.00,1.00


## 2. 划分训练/测试集 + 标准化

80/20 划分；标准化对逻辑回归（基于梯度）必要，对树模型无影响但统一处理无害。`fit_transform` 只在训练集上 fit，防止测试集信息泄漏。

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

## 3. 三模型对比

| 模型 | 原理 | 特点 |
|---|---|---|
| 逻辑回归 | 线性概率模型 | 快、可解释、只能学线性关系 |
| 随机森林 | 多树投票（Bagging） | 抗过拟合、能学非线性 |
| XGBoost | 梯度提升（Boosting） | 逐棵纠错，表格数据王者 |

In [4]:
models = {
    "逻辑回归": LogisticRegression(max_iter=1000),
    "随机森林": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1])
    print(f"{name}: AUC = {auc:.3f}")

逻辑回归: AUC = 0.883


随机森林: AUC = 0.868


XGBoost: AUC = 0.847


> 模拟数据线性关系强，逻辑回归 AUC 最高属正常现象；真实电商数据非线性更强，XGBoost 通常反超。工程上 XGBoost 生态最成熟（特征重要性、调参、部署工具链完善），作为最终模型。

## 4. XGBoost 详细评估

重点看**召回率（recall）**：漏掉一个要流失的用户 = 少一次挽留机会，业务上召回率比精确率更重要。

In [5]:
best = XGBClassifier(eval_metric="logloss", random_state=42)
best.fit(X_train_s, y_train)
print(classification_report(y_test, best.predict(X_test_s)))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       928
           1       0.48      0.43      0.46        72

    accuracy                           0.93      1000
   macro avg       0.72      0.70      0.71      1000
weighted avg       0.92      0.93      0.92      1000



## 5. 特征重要性

Top 特征与业务直觉一致：越久没来（recency 大）、登录越少，越可能流失。

In [6]:
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(3).to_string())

recency             0.273
login_freq          0.149
monetary            0.146
customer_service    0.089
promo_sensitive     0.087
browse_depth        0.087
addcart_rate        0.086
frequency           0.083


## 6. K-Means 用户分群（K=4）

K=4 由业务解释性确定：高价值活跃 / 高价值沉默 / 价格敏感 / 低价值流失，每类对应不同运营策略。

In [7]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
df["cluster"] = km.fit_predict(X)
summary = df.groupby("cluster")[["recency", "frequency", "monetary", "login_freq", "churn"]].mean()
summary.round(2)

,recency,frequency,monetary,login_freq,churn
cluster,,,,,
0,28.42,5.05,446.10,9.98,0.08
1,31.23,5.05,1015.53,10.09,0.02
2,29.93,5.01,706.54,10.01,0.05
3,30.50,4.98,166.01,10.01,0.16


## 7. 高风险用户清单输出

按流失概率降序导出，供运营定向发召回优惠券；线上落地时对比发券组与对照组的回流率即可算 ROI。

In [8]:
df["churn_prob"] = best.predict_proba(sc.transform(X))[:, 1]
df[["user_id", "recency", "frequency", "monetary", "churn", "churn_prob"]].to_csv(
    "predictions.csv", index=False, encoding="utf-8-sig")
print(f"高流失风险用户（概率>0.5）：{(df['churn_prob'] > 0.5).sum()} 人，已导出 predictions.csv")

高流失风险用户（概率>0.5）：434 人，已导出 predictions.csv
